# Capitolul 7: Construirea aplicațiilor de chat
## Pornire rapidă API OpenAI

Acest notebook este adaptat din [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) care include notebook-uri ce accesează serviciile [Azure OpenAI](notebook-azure-openai.ipynb).

API-ul Python OpenAI funcționează și cu modelele Azure OpenAI, cu câteva modificări. Aflați mai multe despre diferențe aici: [Cum să comutați între punctele finale OpenAI și Azure OpenAI cu Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Prezentare generală  
„Modelele mari de limbaj sunt funcții care mapează text în text. Având un șir de text ca intrare, un model mare de limbaj încearcă să prezică textul care va urma”(1). Acest caiet „start rapid” va introduce utilizatorii în concepte la nivel înalt despre LLM, cerințele pachetului de bază pentru a începe cu AML, o introducere blândă în proiectarea prompturilor și câteva exemple scurte de diferite cazuri de utilizare. 


## Cuprins  

[Prezentare generală](#overview)  
[Cum să folosești serviciul OpenAI](#how-to-use-openai-service)  
[1. Crearea serviciului tău OpenAI](#1.-creating-your-openai-service)  
[2. Instalare](#2.-installation)    
[3. Credențiale](#3.-credentials)  

[Cazuri de utilizare](#use-cases)    
[1. Rezumarea textului](#1.-summarize-text)  
[2. Clasificarea textului](#2.-classify-text)  
[3. Generarea de noi nume de produse](#3.-generate-new-product-names)  
[4. Ajustarea fină a unui clasificator](#4.fine-tune-a-classifier)  

[Referințe](#references)


### Construiește primul tău prompt  
Acest exercițiu scurt va oferi o introducere de bază pentru trimiterea de prompturi către un model OpenAI pentru o sarcină simplă de "sumarizare".


**Pași**:  
1. Instalează biblioteca OpenAI în mediul tău python  
2. Încarcă bibliotecile standard de ajutor și setează-ți acreditările de securitate tipice pentru serviciul OpenAI pe care l-ai creat  
3. Alege un model pentru sarcina ta  
4. Creează un prompt simplu pentru model  
5. Trimite cererea ta către API-ul modelului!  


### 1. Instalarea OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importați bibliotecile auxiliare și creați instanța cu acreditări


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Găsirea modelului potrivit  
Modelele precum GPT-4o și GPT-4o mini pot înțelege și genera limbaj natural și sunt disponibile pe platforma OpenAI cu diferite niveluri de putere și viteză potrivite pentru diverse sarcini.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Proiectarea promptului  

„Magia modelelor mari de limbaj constă în faptul că, fiind antrenate să minimizeze această eroare de predicție pe cantități vaste de text, modelele ajung să învețe concepte utile pentru aceste predicții. De exemplu, acestea învață concepte precum”(1):

* cum să scrii corect
* cum funcționează gramatica
* cum să parafrazezi
* cum să răspunzi la întrebări
* cum să porți o conversație
* cum să scrii în multe limbi
* cum să programezi
* etc.

#### Cum să controlezi un model mare de limbaj  
„Dintre toate intrările către un model mare de limbaj, de departe cea mai influentă este promptul de text(1).

Modelele mari de limbaj pot fi stimulate să producă ieșiri în câteva moduri:

Instrucțiune: Spune modelului ce vrei
Completare: Induce modelul să completeze începutul a ceea ce vrei
Demonstrație: Arată modelului ceea ce vrei, cu fie:
Câteva exemple în prompt
Sute sau mii de exemple în setul de date de antrenament pentru reglare fină”



#### Există trei reguli de bază pentru crearea prompturilor:

**Arată și spune**. Fă clar ce vrei fie prin instrucțiuni, exemple sau o combinație a celor două. Dacă dorești ca modelul să clasifice o listă de elemente în ordine alfabetică sau să clasifice un paragraf după sentiment, arată-i că asta dorești.

**Oferă date de calitate**. Dacă încerci să construiești un clasificator sau să faci modelul să urmeze un tipar, asigură-te că există suficiente exemple. Fă-ți timp să corectezi exemplele — modelul este de obicei suficient de inteligent să treacă peste greșelile de ortografie de bază și să-ți ofere un răspuns, dar poate presupune și că acesta este intenționat și poate afecta răspunsul.

**Verifică-ți setările.** Setările temperature și top_p controlează cât de determinist este modelul în generarea unui răspuns. Dacă ceri răspunsuri unde există un singur răspuns corect, atunci vei dori să le setezi mai jos. Dacă vrei răspunsuri mai diverse, le vei seta mai sus. Greșeala numărul unu pe care o fac oamenii cu aceste setări este să presupună că sunt controale pentru „inteligență” sau „creativitate”.


Sursă: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Trimite!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repetă același apel, cum se compară rezultatele?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Rezumarea textului  
#### Provocare  
Rezumă textul adăugând un 'tl;dr:' la sfârșitul unui pasaj de text. Observă cum modelul înțelege cum să efectueze o serie de sarcini fără instrucțiuni suplimentare. Poți experimenta cu indicii mai descriptive decât tl;dr pentru a modifica comportamentul modelului și a personaliza rezumatul pe care îl primești(3).  

Lucrările recente au demonstrat câștiguri substanțiale în multe sarcini și repere NLP prin pre-antrenarea pe un corpus mare de text, urmată de reglare fină pe o sarcină specifică. Deși, în mod tipic, acritic în arhitectură, această metodă necesită totuși seturi de date specifice pentru reglare fină, cu mii sau zeci de mii de exemple. În contrast, oamenii pot, în general, să efectueze o sarcină nouă de limbaj doar din câteva exemple sau instrucțiuni simple - ceva ce sistemele NLP actuale încă se străduiesc să facă în mare măsură. Aici arătăm că scalarea modelelor lingvistice îmbunătățește foarte mult performanța acritic pe puține exemple, uneori ajungând chiar să fie competitivă cu abordările anterioare de reglare fină de ultimă generație.  



Tl;dr  


# Exerciții pentru mai multe cazuri de utilizare  
1. Rezumă textul  
2. Clasifică textul  
3. Generează nume noi de produse


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Clasifică text  
#### Provocare  
Clasifică elementele în categorii furnizate la momentul inferenței. În exemplul următor, oferim atât categoriile, cât și textul de clasificat în prompt (*playground_reference). 

Cerere client: Bună, una dintre tastele de la tastatura laptopului meu s-a rupt recent și am nevoie de o înlocuire:

Categorie clasificată:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generează Nume Noi pentru Produse
#### Provocare
Creează nume de produse pornind de la cuvinte exemplu. Aici includem în prompt informații despre produsul pentru care vom genera nume. De asemenea, oferim un exemplu similar pentru a arăta modelul pe care dorim să-l primim. De asemenea, am setat valoarea temperaturii la un nivel ridicat pentru a crește aleatorietatea și pentru a obține răspunsuri mai inovatoare.

Descriere produs: Un aparat de făcut milkshake acasă
Cuvinte de pornire: rapid, sănătos, compact.
Nume de produse: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descriere produs: O pereche de pantofi care se pot potrivi oricărei mărimi de picior.
Cuvinte de pornire: adaptabil, potrivit, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referințe  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Cele mai bune practici pentru ajustarea fină a modelelor GPT pentru clasificarea textului](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pentru mai mult ajutor  
[Echipa de Comercializare OpenAI](AzureOpenAITeam@microsoft.com) 


# Contribuitori
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
